# Fase 5 — Evaluasi Model IndoBERT (Sentara)

Notebook ini mengevaluasi **best model** hasil Fase 4 pada **test set** (`clean_test.csv`)
yang belum pernah dilihat saat training. Menghasilkan deliverable Checkpoint 5:

- `outputs/reports/evaluation_final.json` — accuracy, precision/recall/F1 macro, per-kelas, confusion matrix
- `outputs/charts/confusion_matrix.png` — heatmap confusion matrix (FR-5.5)
- `outputs/charts/learning_curve.png` — training vs validation loss per epoch (FR-5.6)

Metrik **UTAMA**: macro F1 (target ≥ 0.85). Jalankan di Google Colab (GPU).

## 1. Setup environment

In [ ]:
# Opsi A: clone dari GitHub (ganti URL repo bila perlu)
# !git clone https://github.com/<user>/sistem.git
# %cd sistem

# Opsi B: mount Google Drive lalu cd ke folder proyek
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/SKRIPSI/sistem

!pip -q install "transformers>=4.40.0" "torch>=2.2.0" \n    "scikit-learn>=1.4.0" "pandas>=2.2.0" "matplotlib>=3.9.0"

In [ ]:
import sys
from pathlib import Path

# Pastikan root proyek ada di sys.path agar `import src...` bekerja.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Pastikan best model & test set tersedia

`models/best_model/` (bobot + tokenizer Fase 4) dan `data/processed/clean_test.csv`
harus ada. Bila training dilakukan di sesi terpisah, unggah/mount artefaknya dulu.

In [ ]:
from src.evaluation.config import BEST_MODEL_DIR, EVALUATION_FINAL_JSON
from src.modeling.config import DataConfig

assert BEST_MODEL_DIR.exists(), f'Best model tidak ada di {BEST_MODEL_DIR} — selesaikan Fase 4 dulu.'
test_path = DataConfig().clean_path('test')
assert test_path.exists(), f'Test set tidak ada di {test_path}.'
print('Best model :', BEST_MODEL_DIR)
print('Test set   :', test_path)

## 3. Evaluasi test set (FR-5.1 s/d FR-5.5)

Memuat model, menjalankan inferensi batch pada test set, lalu merakit laporan
final lengkap dengan ringkasan 5-fold CV (Fase 4) dan cek overfitting.

In [ ]:
from src.evaluation.evaluator import load_best_model, predict_split
from src.evaluation.metrics import build_evaluation_report, overfitting_gap
from src.evaluation.cross_val_report import summarize_cv
from src.modeling.data import load_clean_split

test_df = load_clean_split('test')
model, tokenizer = load_best_model()
y_true, y_pred, y_proba = predict_split(test_df, model, tokenizer)

# Ringkasan CV Fase 4 + cek overfitting (isi val_f1/train_f1 dari training log).
cv_summary = summarize_cv()
# Ganti angka berikut dengan F1 macro train & validation dari Fase 4 bila tersedia.
overfit = overfitting_gap(train_f1=cv_summary['f1_macro_mean'], val_f1=cv_summary['f1_macro_mean'])

report = build_evaluation_report(
    y_true, y_pred, cv_summary=cv_summary, overfitting=overfit, write=True,
)
import json
print(json.dumps(report, indent=2, ensure_ascii=False))

## 4. Confusion matrix (FR-5.5)

In [ ]:
from src.evaluation.visualizer import plot_confusion_matrix

cm = report['confusion_matrix']['matrix']
path = plot_confusion_matrix(cm, normalize=False)
print('Tersimpan:', path)

from IPython.display import Image
Image(str(path))

## 5. Learning curve (FR-5.6)

Membaca riwayat loss per epoch dari `trainer_state.json` best model
(fallback `outputs/logs/training_log.csv`).

In [ ]:
from src.evaluation.visualizer import plot_learning_curve

lc_path = plot_learning_curve()
print('Tersimpan:', lc_path)

from IPython.display import Image
Image(str(lc_path))

## 6. Cek gate Fase 5

Validasi: accuracy & macro F1 ≥ 0.85, semua artefak tersedia.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'scripts.validate_phase5_gate'],
                     capture_output=True, text=True).stdout)

## 7. Unduh artefak

In [ ]:
# Arsipkan hasil evaluasi untuk diunduh (jika tidak via Drive)
# !zip -r evaluation_phase5.zip outputs/reports/evaluation_final.json outputs/charts
# from google.colab import files
# files.download('evaluation_phase5.zip')